In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hin-txt/hin_Deva (1).txt
/kaggle/input/hin-noise/direct_noise_data_tgt
/kaggle/input/hin-noise/direct_noise_data_src
/kaggle/input/hi-gec/valid.src
/kaggle/input/hi-gec/train.src
/kaggle/input/hi-gec/valid.tgt
/kaggle/input/hi-gec/train.tgt
/kaggle/input/hi-gec/test.tgt
/kaggle/input/hi-gec/test.src


In [2]:
!pip install "protobuf==3.20.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 3.20.1 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.1 which is incompatible.
google-cloud-secret-manager 2.25.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 3.20.1 which is incompatible.
goo

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Hf-Token")

In [4]:
from huggingface_hub import login
login(secret_value_0)

In [ ]:
with open("/kaggle/input/hi-gec/train.src", "r", encoding="utf-8") as f:
    train_src = f.read().splitlines()

with open("/kaggle/input/hi-gec/train.tgt", "r", encoding="utf-8") as f:
    train_tgt = f.read().splitlines()

with open("/kaggle/input/hi-gec/valid.src", "r", encoding="utf-8") as f:
    valid_src = f.read().splitlines()

with open("/kaggle/input/hi-gec/valid.tgt", "r", encoding="utf-8") as f:
    valid_tgt = f.read().splitlines()

with open("/kaggle/input/hi-gec/test.src", "r", encoding="utf-8") as f:
    test_src = f.read().splitlines()

with open("/kaggle/input/hi-gec/test.tgt", "r", encoding="utf-8") as f:
    test_tgt = f.read().splitlines()

In [ ]:
import torch
from datasets import load_dataset, interleave_datasets, DatasetDict
from transformers import (
    AutoModelForSeq2SeqLM,
    AlbertTokenizer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

PATH_BASE_TRAIN_SRC = "/kaggle/input/hi-gec/train.src"
PATH_BASE_TRAIN_TGT = "/kaggle/input/hi-gec/train.tgt"
PATH_BASE_VALID_SRC = "/kaggle/input/hi-gec/valid.src"
PATH_BASE_VALID_TGT = "/kaggle/input/hi-gec/valid.tgt"
PATH_NOISE_TRAIN_SRC = "/kaggle/input/hin-noise/direct_noise_data_src"
PATH_NOISE_TRAIN_TGT = "/kaggle/input/hin-noise/direct_noise_data_tgt"
MODEL_CHECKPOINT = "ai4bharat/IndicBART"
OUTPUT_DIR = "indicbart-hindi-gec-augmented-v1"


SUBSET_SIZE = 20_000


print("Loading datasets...")

dataset_base = load_dataset('text', data_files={'train': PATH_BASE_TRAIN_SRC})['train']
dataset_base_tgt = load_dataset('text', data_files={'train': PATH_BASE_TRAIN_TGT})['train']
dataset_base = dataset_base.rename_column('text', 'incorrect_text')
dataset_base = dataset_base.add_column('correct_text', dataset_base_tgt['text'])

dataset_noise_full = load_dataset('text', data_files={'train': PATH_NOISE_TRAIN_SRC})['train']
dataset_noise_tgt_full = load_dataset('text', data_files={'train': PATH_NOISE_TRAIN_TGT})['train']
dataset_noise_full = dataset_noise_full.rename_column('text', 'incorrect_text')
dataset_noise_full = dataset_noise_full.add_column('correct_text', dataset_noise_tgt_full['text'])

print(f"Creating a random subset of {SUBSET_SIZE} noise samples...")
dataset_noise_subset = dataset_noise_full.shuffle(seed=42).select(range(SUBSET_SIZE))

print("Combining and interleaving datasets (3:1 ratio)...")
combined_train_dataset = interleave_datasets(
    [dataset_base, dataset_noise_subset], 
    probabilities=[0.75, 0.25], # 3:1 ratio
    seed=42,
    stopping_strategy="all_exhausted" # Oversample the small dataset
)

dataset_valid = load_dataset('text', data_files={'validation': PATH_BASE_VALID_SRC})['validation']
dataset_valid_tgt = load_dataset('text', data_files={'validation': PATH_BASE_VALID_TGT})['validation']
dataset_valid = dataset_valid.rename_column('text', 'incorrect_text')
dataset_valid = dataset_valid.add_column('correct_text', dataset_valid_tgt['text'])

final_datasets = DatasetDict({
    'train': combined_train_dataset,
    'validate': dataset_valid
})

print("Final dataset structure (iterative run):")
print(final_datasets)

print("Loading tokenizer...")
tokenizer = AlbertTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    do_lower_case=False,
    use_fast=False,
    keep_accents=True
)
source_lang_token = '<2hi>'
target_lang_token = '<2hi>'
max_seq_length = 128

def preprocess_function(examples):
    inputs = [f"{ex} </s> {source_lang_token}" for ex in examples['incorrect_text']]
    labels = [f"{target_lang_token} {ex} </s>" for ex in examples['correct_text']]
    model_inputs = tokenizer(
        text=inputs,
        text_target=labels, 
        max_length=max_seq_length, 
        truncation=True,
        padding=False
    )
    return model_inputs

print("Tokenizing all datasets...")
tokenized_datasets = final_datasets.map(
    preprocess_function, 
    batched=True, 
    batch_size=1000,
    remove_columns=final_datasets["train"].column_names
)
print(tokenized_datasets)

2025-11-12 18:21:19.493158: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762971679.660245      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762971679.705250      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Loading datasets...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

... (9 outputs left out)

Tokenizing all datasets...


Map:   0%|          | 0/79908 [00:00<?, ? examples/s]

Map:   0%|          | 0/976 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 79908
    })
    validate: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 976
    })
})


In [ ]:
print("Loading model for fine-tuning...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3, 
    per_device_train_batch_size=8, 
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    learning_rate=3e-5,
    save_total_limit=3, 
    
    eval_strategy="steps",
    eval_steps=200, 
    
    save_strategy="steps",
    save_steps=200, 
    
    load_best_model_at_end=True,         
    metric_for_best_model="eval_loss",   
    greater_is_better=False,             
    
    logging_strategy="steps",
    logging_steps=50,
    report_to=['tensorboard'],
    
    predict_with_generate=True,
    fp16=True, 

    push_to_hub=True,
    hub_model_id="sarthak2314/indicbart-hindi-gec-v1", 
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validate"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print(f"Starting training... Model will be saved to {OUTPUT_DIR}")
trainer.train()

print("Training complete! The trainer has auto-loaded the best model.")
trainer.save_model(f"{OUTPUT_DIR}/best_model")
print(f"Best model saved to {OUTPUT_DIR}/best_model")

Loading model for fine-tuning...


pytorch_model.bin:   0%|          | 0.00/976M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/976M [00:00<?, ?B/s]

/tmp/ipykernel_48/1820450380.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting training... Model will be saved to indicbart-hindi-gec-augmented-v1


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


... (2 outputs left out)

Training complete! The trainer has auto-loaded the best model.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Best model saved to indicbart-hindi-gec-augmented-v1/best_model


In [7]:
!zip -r /kaggle/working/best_model_v1.zip /kaggle/working/indicbart-hindi-gec-augmented-v1/best_model

  adding: kaggle/working/indicbart-hindi-gec-augmented-v1/best_model/ (stored 0%)
  adding: kaggle/working/indicbart-hindi-gec-augmented-v1/best_model/spiece.model (deflated 61%)
  adding: kaggle/working/indicbart-hindi-gec-augmented-v1/best_model/config.json (deflated 57%)
  adding: kaggle/working/indicbart-hindi-gec-augmented-v1/best_model/training_args.bin (deflated 52%)
  adding: kaggle/working/indicbart-hindi-gec-augmented-v1/best_model/tokenizer_config.json (deflated 87%)
  adding: kaggle/working/indicbart-hindi-gec-augmented-v1/best_model/model.safetensors (deflated 7%)
  adding: kaggle/working/indicbart-hindi-gec-augmented-v1/best_model/added_tokens.json (deflated 61%)
  adding: kaggle/working/indicbart-hindi-gec-augmented-v1/best_model/special_tokens_map.json (deflated 58%)
  adding: kaggle/working/indicbart-hindi-gec-augmented-v1/best_model/generation_config.json (deflated 34%)


In [ ]:
import torch
from transformers import AlbertTokenizer, AutoModelForSeq2SeqLM
import difflib
import string  
from typing import List
from pydantic import BaseModel 

class GrammarError(BaseModel):
    id: int
    type: str
    message: str
    original: str
    suggestion: str
    context: str

def find_errors(original: str, corrected: str) -> List[GrammarError]:
    """Compares original and corrected text to find errors."""
    errors = []
    error_id = 1
    
'
    translator = str.maketrans('', '', string.punctuation)
    
    original_clean = original.translate(translator)
    corrected_clean = corrected.translate(translator)
    
    original_words = original_clean.split()
    corrected_words = corrected_clean.split()
    
    matcher = difflib.SequenceMatcher(None, original_words, corrected_words)
    
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':  
            continue
            
        original_chunk = " ".join(original_words[i1:i2])
        context = original  

        if tag == 'replace':
            suggestion_chunk = " ".join(corrected_words[j1:j2])
            message = f"Possible error. Suggestion: '{suggestion_chunk}'"
            error_type = "Grammar"
        
        elif tag == 'delete':
            suggestion_chunk = "[Deletion]"
            message = "Word(s) may be unnecessary."
            error_type = "Deletion"
        
        elif tag == 'insert':
            suggestion_chunk = " ".join(corrected_words[j1:j2])
            original_chunk = f"(after '{original_words[i1-1]}')" if i1 > 0 else "(at start)"
            message = f"Missing word? Suggestion: '{suggestion_chunk}'"
            error_type = "Insertion"
        else:
            continue 

        errors.append(GrammarError(
            id=error_id,
            type=error_type,
            message=message,
            original=original_chunk,
            suggestion=suggestion_chunk,
            context=context
        ))
        error_id += 1
            
    return errors


MODEL_PATH = "./indicbart-hindi-gec-augmented-v1/best_model" 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model from {MODEL_PATH} onto {DEVICE}...")

tokenizer = AlbertTokenizer.from_pretrained(
    MODEL_PATH,
    do_lower_case=False,
    use_fast=False,
    keep_accents=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(DEVICE) 

print("Model loaded successfully!")


test_sentences = [
    "लड़का स्कूल गई।",  # Gender agreement (Original model passed this)
    "मैं घर जा है।",    # Verb agreement (This should be fixed now)
    "टेबल पर किताब हैं।" # Number agreement (This should be fixed now)
]

for sentence in test_sentences:
    print("-" * 30)
    print(f"ORIGINAL:   {sentence}")
    
    inference_text = f"{sentence} </s> <2hi>"
    
    inputs = tokenizer(
        inference_text, 
        return_tensors="pt",
        padding=True,
        max_length=128,
        truncation=True
    ).to(DEVICE)
    
    output_ids = model.generate(
        inputs["input_ids"], 
        max_length=128, 
        num_beams=5,
        early_stopping=True
    )
    
    corrected_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    print(f"CORRECTED:  {corrected_text}")
    
    # 5. Run the diff logic
    found_errors = find_errors(sentence, corrected_text)
    
    print("\nFOUND ERRORS:")
    if not found_errors:
        print("None")
    else:
        for error in found_errors:
            print(f"  - {error.type}: '{error.original}' -> '{error.suggestion}'")
    print("-" * 30, "\n")

Loading model from ./indicbart-hindi-gec-augmented-v1/best_model onto cuda...
Model loaded successfully!
------------------------------
ORIGINAL:   लड़का स्कूल गई।
CORRECTED:  लड़का स्कूल गया।

FOUND ERRORS:
  - Grammar: 'गई।' -> 'गया।'
------------------------------ 

------------------------------
ORIGINAL:   मैं घर जा है।
CORRECTED:  मैं घर जा रहा है।

FOUND ERRORS:
  - Insertion: '(after 'जा')' -> 'रहा'
------------------------------ 

------------------------------
ORIGINAL:   टेबल पर किताब हैं।
CORRECTED:  टेबल पर किताबें हैं।

FOUND ERRORS:
  - Grammar: 'किताब' -> 'किताबें'
------------------------------ 

